# Mini-DINO: Self-Supervised Learning from First Principles

A small-scale implementation of the DINO self-distillation framework designed to run cleanly on free Kaggle/Colab GPUs. We train a Tiny ViT on a 5,000-image subset of CIFAR-10 for 15 epochs, verify EMA and output behaviour, visualise learned embeddings and ViT attention maps, and run an ablation study comparing training with and without the EMA momentum teacher.

```
Image
  |
Multi-crop augmentation (2 global + 4 local)
  |
  +---> Student ViT ---> Student output
  |                          |
  |                      DINO loss
  |                          |
  +---> Teacher ViT ---> Teacher output (global crops only)
            ^
            |
           EMA
```

---
## 1. Environment Setup & Configuration

In [ ]:
import os, copy, math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    torch.cuda.manual_seed_all(42)

os.makedirs('results', exist_ok=True)

# Hyperparameters configured for fast Kaggle/Colab execution
CFG = dict(
    img_size=32,
    patch_size=4,        # 32 / 4 = 8x8 = 64 patches
    embed_dim=192,
    depth=4,
    num_heads=3,
    mlp_ratio=4,
    head_hidden=512,
    head_bottleneck=128,
    head_out=256,
    n_global=2,
    n_local=4,
    batch_size=64,
    epochs=15,
    lr=5e-4,
    weight_decay=0.04,
    teacher_temp=0.04,
    student_temp=0.1,
    momentum_base=0.996,
    center_momentum=0.9,
    num_images=5000,
)

for k, v in CFG.items():
    print(f'  {k:20s} = {v}')

---
## 2. Dataset Preparation (CIFAR-10 Subset)

In [ ]:
# Download CIFAR-10 (on Kaggle make sure Internet is enabled in Settings)
full_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
CLASS_NAMES = full_dataset.classes

# Sample 5,000 images for fast self-supervised training
indices = np.random.choice(len(full_dataset), CFG['num_images'], replace=False)
subset = Subset(full_dataset, indices)
print(f'Loaded {len(subset)} images from CIFAR-10 ({len(CLASS_NAMES)} classes)')

# Plot sample images (labels are only used for evaluation/visualisation, never in training)
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img, label = subset[i]
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample images from CIFAR-10 subset', fontsize=14)
plt.tight_layout()
plt.savefig('results/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Multi-Crop Augmentation

DINO constructs two **global views** (covering 40%--100% of the original image) and several **local views** (covering 5%--40% of the image). All crops are resized back to $32 \times 32$ here for simplicity. Crucially:
- **Teacher** receives only the **global views**.
- **Student** receives **all views** (both global and local).
This forces local-to-global correspondence.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)


class DINOMultiCrop:
    """Generate n_global large-scale crops + n_local small-scale crops."""
    def __init__(self, n_global=2, n_local=4, img_size=32):
        common = [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]
        self.global_t = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.4, 1.0),
                                interpolation=T.InterpolationMode.BICUBIC),
            *common,
        ])
        self.local_t = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.05, 0.4),
                                interpolation=T.InterpolationMode.BICUBIC),
            *common,
        ])
        self.n_global = n_global
        self.n_local = n_local

    def __call__(self, img):
        crops  = [self.global_t(img) for _ in range(self.n_global)]
        crops += [self.local_t(img)  for _ in range(self.n_local)]
        return crops


class DINODataset(Dataset):
    """Wraps a dataset to return multi-crop views."""
    def __init__(self, base, transform):
        self.base = base
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        return self.transform(img), label


def dino_collate(batch):
    crops_list, labels = zip(*batch)
    n_views = len(crops_list[0])
    crops = [torch.stack([c[i] for c in crops_list]) for i in range(n_views)]
    return crops, torch.tensor(labels)


augmentation = DINOMultiCrop(CFG['n_global'], CFG['n_local'], CFG['img_size'])
train_ds = DINODataset(subset, augmentation)
# num_workers=0 avoids multiprocessing issues with custom classes in Kaggle/Windows notebooks
train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    collate_fn=dino_collate,
    num_workers=0,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
)

# Visualise the generated crops for one image
sample_crops, sample_label = train_ds[0]
titles = [f'Global {i+1}' for i in range(CFG['n_global'])] + [
    f'Local {i+1}' for i in range(CFG['n_local'])
]

fig, axes = plt.subplots(1, len(sample_crops), figsize=(2.5 * len(sample_crops), 3))
for ax, crop, title in zip(axes, sample_crops, titles):
    img_np = crop.permute(1, 2, 0).numpy() * np.array(CIFAR_STD) + np.array(CIFAR_MEAN)
    ax.imshow(np.clip(img_np, 0, 1))
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.suptitle(f'Multi-crop views ({CLASS_NAMES[sample_label]})', fontsize=13)
plt.tight_layout()
plt.savefig('results/multicrop_example.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Student & Teacher Tiny ViT + DINO Projection Head

We build a lightweight Vision Transformer (`embed_dim=192`, `depth=4`, `num_heads=3`, `patch_size=4`) paired with a 3-layer MLP projection head with $L_2$ bottleneck normalisation.

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=192):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Attention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, return_attn=False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        return (out, attn) if return_attn else out


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
        )

    def forward(self, x, return_attn=False):
        if return_attn:
            a_out, a_w = self.attn(self.norm1(x), return_attn=True)
            x = x + a_out
            x = x + self.mlp(self.norm2(x))
            return x, a_w
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class TinyViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, embed_dim=192, depth=4, num_heads=3, mlp_ratio=4):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + n_patches, embed_dim))
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x, return_attn=False):
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(x.shape[0], -1, -1), x], dim=1)
        x = x + self.pos_embed
        attn_w = None
        for i, blk in enumerate(self.blocks):
            if return_attn and i == len(self.blocks) - 1:
                x, attn_w = blk(x, return_attn=True)
            else:
                x = blk(x)
        x = self.norm(x)
        cls_out = x[:, 0]
        return (cls_out, attn_w) if return_attn else cls_out


class DINOHead(nn.Module):
    def __init__(self, in_dim, hidden=512, bottleneck=128, out_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, bottleneck),
        )
        self.last = nn.Linear(bottleneck, out_dim, bias=False)

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1)
        return self.last(x)


class DINOModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x, return_attn=False):
        if return_attn:
            feat, attn = self.backbone(x, return_attn=True)
            return self.head(feat), attn
        return self.head(self.backbone(x))


# Sanity-check model shapes and parameter count
_vit = TinyViT(**{k: CFG[k] for k in ['img_size', 'patch_size', 'embed_dim', 'depth', 'num_heads', 'mlp_ratio']})
_head = DINOHead(CFG['embed_dim'], CFG['head_hidden'], CFG['head_bottleneck'], CFG['head_out'])
_m = DINOModel(_vit, _head).to(DEVICE)
_x = torch.randn(2, 3, 32, 32, device=DEVICE)
_o = _m(_x)
print(f'Input shape : {_x.shape}')
print(f'Output shape: {_o.shape}')
print(f'Total params: {sum(p.numel() for p in _m.parameters()) / 1e6:.2f} M')
del _vit, _head, _m, _x, _o

---
## 5. DINO Loss, Centering, Sharpening & EMA Update

To avoid representation collapse without negative pairs, DINO uses:
1. **Centering** ($t - c$): prevents one dimension from dominating (avoids collapse to a one-hot vector).
2. **Sharpening** ($\tau_t < \tau_s$): uses a low teacher temperature ($\tau_t = 0.04$) to push the distribution away from uniform.
3. **EMA Teacher** ($\theta_t \leftarrow \lambda \theta_t + (1 - \lambda)\theta_s$): provides a stable target.

In [ ]:
def dino_loss(s_outs, t_outs, center, s_temp, t_temp):
    """Cross-entropy between centered+sharpened teacher and softened student across views."""
    total = 0.0
    n_terms = 0
    for ti, t_out in enumerate(t_outs):
        t_probs = F.softmax((t_out - center) / t_temp, dim=-1)
        for si, s_out in enumerate(s_outs):
            if ti == si:
                continue  # skip same crop index
            s_logp = F.log_softmax(s_out / s_temp, dim=-1)
            total -= (t_probs * s_logp).sum(dim=-1).mean()
            n_terms += 1
    return total / n_terms


@torch.no_grad()
def ema_update(student, teacher, momentum):
    """Momentum update of teacher parameters from student parameters."""
    for sp, tp in zip(student.parameters(), teacher.parameters()):
        tp.data.mul_(momentum).add_((1.0 - momentum) * sp.data)


def cosine_schedule(base, final, steps):
    t = np.arange(steps)
    return final + 0.5 * (base - final) * (1 + np.cos(np.pi * t / steps))


print('DINO loss and EMA helper functions ready.')

---
## 6. Training Pipeline

In [ ]:
def train_mini_dino(loader, cfg, use_ema=True, device=DEVICE, tag=''):
    vit_kw = {k: cfg[k] for k in ['img_size', 'patch_size', 'embed_dim', 'depth', 'num_heads', 'mlp_ratio']}
    backbone = TinyViT(**vit_kw)
    head = DINOHead(cfg['embed_dim'], cfg['head_hidden'], cfg['head_bottleneck'], cfg['head_out'])
    student = DINOModel(backbone, head).to(device)
    teacher = copy.deepcopy(student).to(device)
    for p in teacher.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(student.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    total_steps = cfg['epochs'] * len(loader)
    lr_sched  = cosine_schedule(cfg['lr'], 1e-6, total_steps)
    mom_sched = cosine_schedule(cfg['momentum_base'], 1.0, total_steps)
    center = torch.zeros(cfg['head_out'], device=device)

    loss_history = []
    step = 0
    ema_tag = 'EMA' if use_ema else 'no-EMA'

    for epoch in range(cfg['epochs']):
        ep_losses = []
        pbar = tqdm(loader, desc=f'{tag}Ep {epoch+1}/{cfg["epochs"]} ({ema_tag})')
        for crops, _ in pbar:
            crops = [c.to(device) for c in crops]

            for pg in optimizer.param_groups:
                pg['lr'] = lr_sched[step]

            # Teacher forward pass (global crops only, no grad)
            with torch.no_grad():
                t_in = torch.cat(crops[:cfg['n_global']])
                t_out = teacher(t_in)
                t_outs = t_out.chunk(cfg['n_global'])

            # Student forward pass (all crops)
            s_in = torch.cat(crops)
            s_out = student(s_in)
            s_outs = s_out.chunk(cfg['n_global'] + cfg['n_local'])

            loss = dino_loss(s_outs, t_outs, center, cfg['student_temp'], cfg['teacher_temp'])

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 3.0)
            optimizer.step()

            # Update teacher & running center
            with torch.no_grad():
                if use_ema:
                    ema_update(student, teacher, mom_sched[step])
                else:
                    for sp, tp in zip(student.parameters(), teacher.parameters()):
                        tp.data.copy_(sp.data)
                batch_center = torch.cat(t_outs).mean(dim=0)
                center.mul_(cfg['center_momentum']).add_((1.0 - cfg['center_momentum']) * batch_center)

            lv = loss.item()
            ep_losses.append(lv)
            loss_history.append(lv)
            pbar.set_postfix(loss=f'{lv:.4f}')
            step += 1

        print(f'  Epoch {epoch+1:2d} | avg loss: {np.mean(ep_losses):.4f} | lr: {lr_sched[step-1]:.6f}')

    return dict(loss=loss_history, student=student, teacher=teacher, center=center)

In [ ]:
print('=' * 60)
print('TRAINING MINI-DINO (WITH EMA)')
print('=' * 60)
results = train_mini_dino(train_loader, CFG, use_ema=True)
print('\nTraining complete!')

---
## 7. Verification of Student/Teacher Outputs & EMA

We verify that:
1. Student and teacher produce distinct outputs on the same batch.
2. Teacher weights lag behind student weights smoothly via EMA.

In [ ]:
student = results['student']
teacher = results['teacher']

test_crops, _ = next(iter(train_loader))
test_img = test_crops[0][:4].to(DEVICE)

with torch.no_grad():
    s_out = student(test_img)
    t_out = teacher(test_img)

print(f'Student output shape: {s_out.shape}')
print(f'Teacher output shape: {t_out.shape}')
print(f'Student -- mean: {s_out.mean():.4f}, std: {s_out.std():.4f}')
print(f'Teacher -- mean: {t_out.mean():.4f}, std: {t_out.std():.4f}')
diff = (s_out - t_out).abs().mean().item()
print(f'\nMean |student_out - teacher_out|: {diff:.4f}')

s_p = list(student.parameters())
t_p = list(teacher.parameters())
diffs = [(s.data - t.data).abs().mean().item() for s, t in zip(s_p, t_p)]
print(f'Parameter difference across {len(diffs)} tensors:')
print(f'  Mean: {np.mean(diffs):.6f} | Max: {np.max(diffs):.6f}')
print('EMA check:', 'PASSED (teacher is distinct from student)' if np.mean(diffs) > 1e-6 else 'WARNING')

---
## 8. Plot Training Loss

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results['loss'], alpha=0.35, color='steelblue', lw=0.8)
ax1.set_xlabel('Step')
ax1.set_ylabel('DINO Loss')
ax1.set_title('Raw Step Loss')

w = max(len(results['loss']) // 50, 1)
sm = np.convolve(results['loss'], np.ones(w) / w, mode='valid')
ax2.plot(sm, color='steelblue', lw=2)
ax2.set_xlabel('Step')
ax2.set_ylabel('Smoothed Loss')
ax2.set_title(f'Moving Average Loss (window={w})')

plt.suptitle('Mini-DINO Training Loss', fontsize=14)
plt.tight_layout()
plt.savefig('results/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Extract & Visualise Learned Embeddings (PCA & t-SNE)

We extract the 192-dimensional CLS token from the backbone for all 5,000 images and project them into 2D using PCA and t-SNE, colouring points by their ground-truth CIFAR-10 class.

In [ ]:
eval_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
student.eval()

embeddings, all_labels = [], []
for i in tqdm(range(len(subset)), desc='Extracting CLS embeddings'):
    img, label = subset[i]
    with torch.no_grad():
        feat = student.backbone(eval_tf(img).unsqueeze(0).to(DEVICE)).cpu()
    embeddings.append(feat.squeeze(0))
    all_labels.append(label)

embeddings = torch.stack(embeddings).numpy()
all_labels = np.array(all_labels)
print(f'Extracted embeddings shape: {embeddings.shape}')

pca_2d  = PCA(n_components=2).fit_transform(embeddings)
tsne_2d = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=42).fit_transform(embeddings)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
for c in range(10):
    m = all_labels == c
    ax1.scatter(pca_2d[m, 0],  pca_2d[m, 1],  s=10, alpha=0.6, label=CLASS_NAMES[c])
    ax2.scatter(tsne_2d[m, 0], tsne_2d[m, 1], s=10, alpha=0.6, label=CLASS_NAMES[c])
ax1.set_title('PCA of Learned CLS Embeddings', fontsize=13)
ax1.legend(fontsize=8, markerscale=2)
ax2.set_title('t-SNE of Learned CLS Embeddings', fontsize=13)
ax2.legend(fontsize=8, markerscale=2)
plt.suptitle(f'Mini-DINO Self-Supervised Embeddings ({CFG["num_images"]} images, {CFG["epochs"]} epochs)', fontsize=14)
plt.tight_layout()
plt.savefig('results/embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Visualise ViT Attention Maps

A signature property of DINO is that self-attention maps from the final ViT block naturally highlight foreground objects without ever seeing segmentation or class labels.

In [ ]:
grid_size = CFG['img_size'] // CFG['patch_size']  # 8x8 patch grid
student.eval()

n_show = 8
show_idx = np.random.choice(len(subset), n_show, replace=False)
fig, axes = plt.subplots(3, n_show, figsize=(2.5 * n_show, 8))

for col, idx in enumerate(show_idx):
    img_pil, label = subset[idx]
    with torch.no_grad():
        _, attn = student.backbone(eval_tf(img_pil).unsqueeze(0).to(DEVICE), return_attn=True)
    # CLS token attention to patch tokens (shape: num_heads x 64)
    cls_attn = attn[0, :, 0, 1:].mean(0).reshape(grid_size, grid_size).cpu().numpy()
    cls_attn = np.kron(cls_attn, np.ones((CFG['patch_size'], CFG['patch_size'])))
    img_np = np.array(img_pil) / 255.0

    axes[0, col].imshow(img_np)
    axes[0, col].set_title(CLASS_NAMES[label], fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(cls_attn, cmap='inferno')
    axes[1, col].axis('off')

    axes[2, col].imshow(img_np)
    axes[2, col].imshow(cls_attn, cmap='inferno', alpha=0.5)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('Image', fontsize=12)
axes[1, 0].set_ylabel('Attention', fontsize=12)
axes[2, 0].set_ylabel('Overlay', fontsize=12)
plt.suptitle('CLS Token Attention Maps (Last Layer, Head-Averaged)', fontsize=14)
plt.tight_layout()
plt.savefig('results/attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-head attention breakdown on a single image
img_pil, label = subset[0]
with torch.no_grad():
    _, attn = student.backbone(eval_tf(img_pil).unsqueeze(0).to(DEVICE), return_attn=True)

n_heads = CFG['num_heads']
fig, axes = plt.subplots(1, n_heads + 2, figsize=(3 * (n_heads + 2), 3))
axes[0].imshow(np.array(img_pil) / 255.0)
axes[0].set_title('Original')
axes[0].axis('off')

for h in range(n_heads):
    ha = attn[0, h, 0, 1:].reshape(grid_size, grid_size).cpu().numpy()
    ha = np.kron(ha, np.ones((CFG['patch_size'], CFG['patch_size'])))
    axes[h + 1].imshow(ha, cmap='inferno')
    axes[h + 1].set_title(f'Head {h+1}')
    axes[h + 1].axis('off')

avg = attn[0, :, 0, 1:].mean(0).reshape(grid_size, grid_size).cpu().numpy()
avg = np.kron(avg, np.ones((CFG['patch_size'], CFG['patch_size'])))
axes[-1].imshow(avg, cmap='inferno')
axes[-1].set_title('Head Average')
axes[-1].axis('off')
plt.suptitle(f'Per-Head CLS Attention ({CLASS_NAMES[label]})', fontsize=13)
plt.tight_layout()
plt.savefig('results/per_head_attention.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Ablation Study: With EMA vs Without EMA

What happens if we remove the momentum teacher and simply copy student weights directly into the teacher at every step (`momentum = 0`)? We run the exact same setup with `use_ema=False` and compare loss trajectories and embedding quality.

In [ ]:
print('=' * 60)
print('ABLATION: TRAINING WITHOUT EMA (TEACHER = STUDENT)')
print('=' * 60)
results_no_ema = train_mini_dino(train_loader, CFG, use_ema=False, tag='[no-EMA] ')

# Compare smoothed training loss
fig, ax = plt.subplots(figsize=(10, 5))
w = max(len(results['loss']) // 50, 1)
sm_ema = np.convolve(results['loss'],        np.ones(w) / w, mode='valid')
sm_ne  = np.convolve(results_no_ema['loss'], np.ones(w) / w, mode='valid')
ax.plot(sm_ema, label='With EMA (momentum=0.996->1.0)', lw=2)
ax.plot(sm_ne,  label='Without EMA (direct copy)', lw=2, ls='--')
ax.set_xlabel('Step')
ax.set_ylabel('Smoothed Loss')
ax.set_title('Ablation: Effect of EMA Momentum Teacher on Training Loss')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('results/ablation_loss.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare t-SNE embeddings
student_ne = results_no_ema['student']
student_ne.eval()
emb_ne = []
for i in tqdm(range(len(subset)), desc='Extracting No-EMA embeddings'):
    img, _ = subset[i]
    with torch.no_grad():
        feat = student_ne.backbone(eval_tf(img).unsqueeze(0).to(DEVICE)).cpu()
    emb_ne.append(feat.squeeze(0))
emb_ne = torch.stack(emb_ne).numpy()

tsne_ne = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=42).fit_transform(emb_ne)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
for c in range(10):
    m = all_labels == c
    ax1.scatter(tsne_2d[m, 0], tsne_2d[m, 1], s=10, alpha=0.6, label=CLASS_NAMES[c])
    ax2.scatter(tsne_ne[m, 0], tsne_ne[m, 1], s=10, alpha=0.6, label=CLASS_NAMES[c])
ax1.set_title('With EMA Teacher', fontsize=13)
ax1.legend(fontsize=8, markerscale=2)
ax2.set_title('Without EMA Teacher', fontsize=13)
ax2.legend(fontsize=8, markerscale=2)
plt.suptitle('Ablation: t-SNE of Learned CLS Embeddings', fontsize=14)
plt.tight_layout()
plt.savefig('results/ablation_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
torch.save(results['student'].state_dict(), 'results/student_ema.pth')
torch.save(results_no_ema['student'].state_dict(), 'results/student_no_ema.pth')
print('Saved checkpoints and figures to ./results/')